# 09 — Tốc độ suy luận & triển khai biên

**Chủ: Phong Trần** · lý thuyết: `docs/LY_THUYET.md` 6.4–6.5 · code: `src/gtsrb/deploy/speed.py`

Câu hỏi của notebook này: **gắn lên xe thật thì chọn model nào?**

Chạy `make speed` trước.

In [ ]:
# Cho notebook thấy src/.  Chạy ô này đầu tiên.
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)          # mọi đường dẫn trong repo là tương đối từ gốc

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from gtsrb import CLASS_NAMES, NUM_CLASSES
print("torch", torch.__version__, "| MPS", torch.backends.mps.is_available(),
      "| CUDA", torch.cuda.is_available())

## 1. ★ Đo latency cho ĐÚNG — sai một bước là mọi kết luận vô giá trị

| Bước | Vì sao |
|---|---|
| `model.eval()` + `torch.no_grad()` | tắt dropout/BN-train, không dựng graph autograd |
| ≥ 20 vòng **warm-up** | lần chạy đầu tốn thời gian khởi tạo kernel, cấp phát bộ nhớ, nạp cache |
| ★ **Đồng bộ thiết bị trước VÀ sau khi bấm giờ** | GPU/MPS chạy **bất đồng bộ**: `model(x)` trả về ngay khi xếp hàng xong, **chưa tính xong**. Không sync thì đang đo thời gian **gửi lệnh** — sai hàng chục lần. Đây là lỗi benchmark phổ biến nhất |
| 100 vòng đo, báo **p50 và p95** | hệ thống thời gian thực quan tâm **trường hợp xấu**; mean che mất đuôi phân phối |
| batch=1 **và** batch=64 | batch=1 là tình huống xe thật (từng khung ảnh); batch=64 đo throughput |
| ghi rõ **thiết bị** và phiên bản thư viện | số latency **không so sánh được** giữa hai máy khác nhau |

In [ ]:
# Minh hoạ: đo SAI (không sync) vs đo ĐÚNG (có sync)
import time
from gtsrb.models.registry import build_model
from gtsrb.utils.seed import pick_device

device = pick_device("auto")
m = build_model("m2_vggres", img_size=48).to(device).eval()
x = torch.randn(64, 3, 48, 48, device=device)

def sync():
    if device.type == "cuda": torch.cuda.synchronize()
    elif device.type == "mps": torch.mps.synchronize()

with torch.no_grad():
    for _ in range(20): m(x)       # warm-up
    sync()

    # CÁCH SAI: không đồng bộ
    t0 = time.perf_counter()
    for _ in range(50): m(x)
    wrong = (time.perf_counter() - t0) / 50 * 1000

    # CÁCH ĐÚNG: đồng bộ trước và sau
    sync(); t0 = time.perf_counter()
    for _ in range(50): m(x)
    sync()
    right = (time.perf_counter() - t0) / 50 * 1000

print(f"thiết bị: {device}")
print(f"đo KHÔNG sync : {wrong:7.3f} ms/batch")
print(f"đo CÓ sync    : {right:7.3f} ms/batch")
print(f"-> chênh {right/wrong:.2f} lần" if wrong > 0 else "")
print("\n(Trên CPU hai số bằng nhau vì CPU đồng bộ sẵn. Trên GPU/MPS thì khác rõ.)")

## 2. Bảng tốc độ

In [ ]:
path = pathlib.Path("reports/tables/speed.csv")
if not path.exists():
    print("Chưa có bảng. Chạy:", "make speed")
    speed = pd.DataFrame()      # để các ô sau tự bỏ qua, không vỡ notebook
else:
    speed = pd.read_csv(path)
    cols = [c for c in ("model","params_m","flops_g","size_mb","test_top1") if c in speed.columns]
    cols += [c for c in speed.columns if c.endswith("_p50") or c.endswith("_p95")]
    print(speed[cols].to_string(index=False))


## 3. ★ FLOPs KHÔNG PHẢI latency — bài học quan trọng nhất của phần này

In [ ]:
if speed.empty:
    print("chưa có số liệu — chạy make speed")
else:
    lat_col = next((c for c in speed.columns if c.endswith("_bs1_p50")), None)
    if lat_col and speed["flops_g"].notna().any():
        s = speed.dropna(subset=["flops_g", lat_col]).copy()
        s["ms per GFLOP"] = s[lat_col] / s["flops_g"]
        print(s[["model", "flops_g", lat_col, "ms per GFLOP"]].to_string(index=False))
        print()
        print("Nếu latency tỉ lệ với FLOPs thì cột 'ms per GFLOP' phải BẰNG NHAU.")
        if len(s) >= 2:
            spread = s["ms per GFLOP"].max() / s["ms per GFLOP"].min()
            print(f"Thực tế nó chênh {spread:.1f} lần -> latency KHÔNG tỉ lệ với FLOPs.")

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].scatter(s["flops_g"], s[lat_col], s=90)
        for r in s.itertuples(index=False):
            axes[0].annotate(r.model, (r.flops_g, getattr(r, lat_col)),
                             textcoords="offset points", xytext=(7, 4), fontsize=8)
        axes[0].set_xlabel("GFLOPs"); axes[0].set_ylabel(f"latency {lat_col} (ms)")
        axes[0].set_title("FLOPs vs latency thực tế"); axes[0].grid(alpha=.3)
        axes[1].bar(s["model"], s["ms per GFLOP"], color="indianred")
        axes[1].set_ylabel("ms / GFLOP")
        axes[1].set_title("Nếu FLOPs = latency thì các cột phải bằng nhau")
        axes[1].tick_params(axis="x", rotation=20)
        plt.tight_layout(); plt.show()
    else:
        print("Thiếu cột flops_g hoặc latency. Cài thop: pip install thop")

**Vì sao FLOPs không dự đoán được latency — ba nguyên nhân:**

1. **Arithmetic intensity thấp.** FLOPs chỉ đếm **phép tính**, không đếm **truy cập bộ nhớ**.
   Depthwise conv (MobileNetV2) làm rất ít phép tính trên mỗi byte đọc vào → nó bị chặn
   bởi **băng thông bộ nhớ**, không bởi năng lực tính toán. Giảm FLOPs **không giúp gì**
   khi nút cổ chai là bộ nhớ.
2. **Nhiều lần launch kernel.** MobileNetV2 có nhiều lớp mảnh hơn; mỗi lớp là một lần
   launch kernel với overhead cố định.
3. **Mức độ tối ưu của thư viện.** cuDNN/BLAS tối ưu conv dày đặc cực kỹ trong nhiều năm;
   depthwise conv ít được chăm hơn.

**Kết luận hành động: muốn nói về triển khai thì PHẢI đo wall-clock trên thiết bị đích.**

## 4. p50 vs p95 — vì sao hệ thống thời gian thực cần p95

In [ ]:
if speed.empty:
    print("chưa có số liệu — chạy make speed")
else:
    p50c = [c for c in speed.columns if c.endswith("_bs1_p50")]
    p95c = [c for c in speed.columns if c.endswith("_bs1_p95")]
    if p50c and p95c:
        t = speed[["model"] + p50c + p95c].copy()
        t["p95/p50"] = (t[p95c[0]] / t[p50c[0]]).round(2)
        print(t.to_string(index=False))
        print()
        print("p95/p50 càng lớn = latency càng BẤP BÊNH.")
        print("Xe chạy 50 km/h đi được ~1,4 cm mỗi ms. Một khung ảnh chậm gấp đôi")
        print("có thể là khác biệt giữa phanh kịp và không kịp -> phải tính theo p95,")
        print("không phải theo trung bình.")

## 5. ★ Pareto accuracy–latency: gắn lên xe thật thì chọn gì?

In [ ]:
pareto_png = pathlib.Path("reports/figures/pareto.png")
if pareto_png.exists():
    import matplotlib.image as mpimg
    fig, ax = plt.subplots(figsize=(9, 6.5))
    ax.imshow(mpimg.imread(pareto_png)); ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("Chưa có pareto.png. Chạy: make eval && make speed")

## 6. Khuyến nghị triển khai — quyết định trên BA trục, không chỉ accuracy

Phát biểu theo mẫu này, với số thật thay vào:

> Với ngân sách latency **X ms** ở batch=1 trên **\<thiết bị\>**, nhóm chọn **M?** vì:
>
> 1. **Accuracy:** chênh lệch so với model tốt nhất là **?** điểm, và McNemar cho
>    p = **?** → *(có / không)* có ý nghĩa thống kê.
> 2. **Latency p95:** **?** ms, so với **?** ms của model tốt nhất → nhanh hơn **?** lần.
> 3. **Robustness:** relative robustness trung bình **?**, so với **?**.
>
> Trả thêm **?** lần latency để lấy **?** điểm accuracy *không có ý nghĩa thống kê*
> là lựa chọn tồi trên hệ thống thời gian thực.

**Hướng mở rộng nếu còn thời gian:** dynamic quantization int8, export ONNX/CoreML/TFLite,
rồi **đo lại** — chỉ khi đó phần triển khai biên mới thành kết luận thực sự thay vì
so sánh tương đối.